## Exemplo Introdutório de Redes Neurais Convolucionais (CNNs)

As Redes Neurais Convolucionais (CNNs) são uma classe de redes neurais profundas, mais comumente aplicadas à análise de imagens, mas também eficazes em outras tarefas como processamento de linguagem natural. Elas são projetadas para processar dados de pixel que têm uma topologia de grade conhecida, como imagens bidimensionais.

Este exemplo demonstrará uma CNN básica para classificar imagens do dataset Fashion MNIST.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

### 1. Carregar e Pré-processar o Dataset Fashion MNIST

Vamos usar o dataset Fashion MNIST, que consiste em 70.000 imagens em tons de cinza de 28x28 pixels, de 10 categorias de itens de vestuário. 60.000 imagens são para treinamento e 10.000 para teste.

In [ ]:
# Carregar o dataset Fashion MNIST
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

# Normalizar as imagens para o intervalo [0, 1]
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

# Redimensionar as imagens para o formato (largura, altura, canais)
# Para imagens em tons de cinza, o número de canais é 1.
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))


class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Formato dos dados de treinamento: {train_images.shape}")
print(f"Formato dos rótulos de treinamento: {train_labels.shape}")
print(f"Formato dos dados de teste: {test_images.shape}")
print(f"Formato dos rótulos de teste: {test_labels.shape}")

### 2. Visualizar algumas imagens

Vamos dar uma olhada nas primeiras 25 imagens do conjunto de treinamento e verificar seus rótulos.

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i].reshape(28,28), cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

### 3. Construir o Modelo CNN

Um modelo CNN típico consiste em:
- **Camadas Convolucionais (`Conv2D`):** Aplicam filtros para extrair características das imagens.
- **Camadas de Pooling (`MaxPooling2D`):** Reduzem a dimensionalidade das características, tornando o modelo mais robusto a pequenas variações e reduzindo o número de parâmetros.
- **Camadas Densas (`Dense`):** As camadas totalmente conectadas no final, usadas para a classificação final.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])

model.summary()

### 4. Compilar e Treinar o Modelo

Após definir a arquitetura do modelo, precisamos compilá-lo, especificando:
- **Otimizador:** Como o modelo será atualizado com base nos dados de treinamento (`'adam'` é um bom otimizador padrão).
- **Função de Perda:** Como o erro será medido (`SparseCategoricalCrossentropy` é adequada para classificação multiclasse quando os rótulos são inteiros).
- **Métricas:** Quais métricas serão monitoradas durante o treinamento e avaliação (`'accuracy'` é a mais comum para classificação).

Em seguida, treinamos o modelo usando os dados de treinamento.

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=10,
                    validation_data=(test_images, test_labels))

### 5. Avaliar o Modelo

Após o treinamento, avaliamos o modelo no conjunto de teste para ver seu desempenho em dados que ele não viu durante o treinamento.

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.show()

test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(f"\nAcurácia no conjunto de teste: {test_acc:.4f}")

### 6. Fazer Predições

Finalmente, podemos usar o modelo treinado para fazer predições em novas imagens.

In [ ]:
probability_model = tf.keras.Sequential([model, tf.keras.layers.Softmax()])
predictions = probability_model.predict(test_images)

def plot_image(i, predictions_array, true_label, img):
  true_label, img = true_label[i], img[i]
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])

  plt.imshow(img.reshape(28,28), cmap=plt.cm.binary)

  predicted_label = np.argmax(predictions_array)
  if predicted_label == true_label:
    color = 'blue'
  else:
    color = 'red'

  plt.xlabel(f"{class_names[predicted_label]} {100*np.max(predictions_array):.2f}% ({class_names[true_label]})",
             color=color)

def plot_value_array(i, predictions_array, true_label):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(10))
  plt.yticks([])
  thisplot = plt.bar(range(10), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')


num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i, predictions[i], test_labels, test_images)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i, predictions[i], test_labels)
plt.tight_layout()
plt.show()